# CaptionBench: the grounding-gap study in one notebook

This notebook runs the whole CSE434 project on a Colab GPU.
It needs no Google Drive, no account, no API key, and no uploaded file.

1. Choose **Runtime > Change runtime type > T4 GPU** (or a larger GPU).
2. Choose **Runtime > Run all**.

The notebook copies the public code, downloads Flickr8k, trains and scores every model, and shows the tables, the figures, and a page of real captions.
Every result stays in the Colab runtime. The last cell downloads a zip file to your computer.

| Profile | What runs | Approximate time on a T4 |
|---|---|---|
| `quick` | One small decoder, zero-shot BLIP base, a short fine-tune. A wiring check. Its scores are not study results. | 15 minutes |
| `midterm` | The conditions in the midterm report: 6 decoders, BLIP base and large, 3 fine-tuned BLIP base seeds. | 2.5 hours |
| `full` | `midterm`, plus a wider decoder on ConvNeXt features and a fine-tuned BLIP large. | 4 hours |

The times are estimates from a local RTX 5070 run. A Colab run has not timed them.
If Colab disconnects, run all cells again. Finished stages are skipped while the runtime still holds its files.
A new runtime starts from the beginning, because nothing is stored outside the runtime.

In [ ]:
# @title Settings
PROFILE = "quick"  # @param ["quick", "midterm", "full"]
FINETUNE_SEEDS = "17 29 43"  # @param {type:"string"}
REPO_REF = "midterm-checkin"  # @param {type:"string"}
REPO_URL = "https://github.com/jhunter11/cse434-fp.git"

In [ ]:
# @title Get the code and check the GPU
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    REPO = Path("/content/cse434-fp")
    if not REPO.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[models]", "matplotlib", "pandas"],
                   check=True)
else:
    REPO = Path.cwd() if (Path.cwd() / "scripts" / "run_study.py").is_file() else Path.cwd().parent
os.chdir(REPO)
os.environ["PYTHONPATH"] = str(REPO / "src")
WORK = REPO / "outputs" / f"study-{PROFILE}"

import torch
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print(f"GPU: {properties.name}, {properties.total_memory / 2**30:.1f} GiB. torch {torch.__version__}")
    if PROFILE == "full" and properties.total_memory < 15 * 2**30:
        print("The full profile fine-tunes BLIP large. Use a GPU with 16 GiB or more, or choose midterm.")
else:
    raise SystemExit("No GPU is connected. Choose Runtime > Change runtime type > T4 GPU, then run all cells again.")
print("Code:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

## Run the study

This cell does all the work. It prints one `===` line when a stage starts and one when the stage ends.
The first stage downloads Flickr8k (1.1 GiB) from a public mirror and checks the SHA256 of each archive.
The 1,000 test images stay sealed. No stage reads them.

In [ ]:
# @title Download the data, train, and score
command = [sys.executable, "-u", "scripts/run_study.py", "--profile", PROFILE, "--work", str(WORK),
           "--finetune-seeds", *FINETUNE_SEEDS.split()]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO)
tail = []
for line in process.stdout:
    tail = (tail + [line])[-40:]
    if line.startswith(("===", "epoch", "{", "Study", "Tables", "Aggregate", "Gallery", "The quick", "  Flickr",
                        "Downloading", "Extracted")):
        print(line, end="")
if process.wait() != 0:
    print("".join(tail))
    raise SystemExit("A stage failed. The lines above show the end of its output.")

## Results

**own** is the score of a caption on its own image. **blind %** is the share of that score that the caption keeps on a different image.
A blind share of 100% means that the metric cannot tell whether the model saw the image.
BLEU-4 and METEOR are multiplied by 100. CIDEr-D runs from 0 to 10. CLIPScore runs from 0 to 2.5.

In [ ]:
# @title Scores on the 200 confirmation images
import json
import pandas as pd
from IPython.display import Image, display

summary = json.loads((WORK / "results" / "summary.json").read_text(encoding="utf-8"))
names = {"bleu4_corpus": ("BLEU-4", 100, 1), "meteor": ("METEOR", 100, 1), "cider_d": ("CIDEr-D", 1, 2),
         "clipscore": ("CLIPScore", 1, 2)}
rows = []
for family, entry in summary["families"].items():
    row = {"condition": family, "runs": entry["runs"]}
    for key, (label, scale, digits) in names.items():
        cell = entry["metrics"][key]
        row[f"{label} own"] = round(scale * cell["matched"], digits)
        row[f"{label} blind %"] = round(100 * cell["blind_share"])
    row["tokens"] = round(entry["caption_tokens_mean"], 1)
    rows.append(row)
display(pd.DataFrame(rows).set_index("condition"))
if PROFILE == "quick":
    print("Quick profile: these scores come from short training runs. They are not study results.")

In [ ]:
# @title Figures
for name in ("blind-share", "finetune-split", "decoder-curves"):
    path = WORK / "report" / "figures" / f"{name}.png"
    if path.is_file():
        display(Image(filename=str(path), width=900 if name == "blind-share" else 480))

In [ ]:
# @title Does the choice of the other image matter?
rules = {"cyclic shift": "grounding-gap.json", "nearest photo": "grounding-gap-nearest.json",
         "farthest photo": "grounding-gap-farthest.json"}
table = {}
for rule, filename in rules.items():
    path = WORK / "results" / filename
    if path.is_file():
        models = json.loads(path.read_text(encoding="utf-8"))["models"]
        table[rule] = {model: round(100 * entry["metrics"]["cider_d"]["blind_share"]) for model, entry in models.items()}
if len(table) > 1:
    print("CIDEr-D blind share (%) under three rules for the other image")
    display(pd.DataFrame(table))
else:
    print("The quick profile uses the cyclic shift only. Run the midterm profile for the other two rules.")

## Real captions

The page shows 16 confirmation photos drawn with a fixed random seed. Nobody chose a photo by its result. Each caption is unedited.
The photos belong to their Flickr owners. Clear this output before you publish the notebook.

In [ ]:
# @title Caption gallery
display(Image(filename=str(WORK / "gallery.png"), width=1000))

## Download the results

The zip file holds the aggregate scores, the figures, the results table, and each model's captions.
It holds no Flickr8k photo and no reference caption, unless you include the gallery.

In [ ]:
# @title Zip and download
INCLUDE_GALLERY = False  # @param {type:"boolean"}
import zipfile
archive = REPO / "outputs" / f"captionbench-{PROFILE}-results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zipped:
    for folder in ("results", "report"):
        for path in sorted((WORK / folder).rglob("*")):
            if path.is_file():
                zipped.write(path, path.relative_to(WORK))
    for path in sorted(WORK.glob("*.jsonl")):
        zipped.write(path, Path("predictions") / path.name)
    for path in sorted(WORK.glob("runs/*/*.json")):
        zipped.write(path, path.relative_to(WORK))
    if INCLUDE_GALLERY:
        zipped.write(WORK / "gallery.pdf", "gallery.pdf")
print(archive, f"{archive.stat().st_size / 2**20:.1f} MiB")
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))